## Powderday SED Workflow — PSB Galaxy Sample

End-to-end pipeline: filter particle files → write Slurm masters → run Powderday RT → extract multi-band photometric fluxes.

The **same** source list drives two runs that differ only in dust treatment:

- **dust_on**  — full dust model: SIMBA live dust masses (`dust_grid_type = 'manual'`, `n_photons_raytracing_dust = 1e7`)
- **dust_off** — dust blocked (`dust_grid_type = 'dtm'`, `dusttometals_ratio = 1e-12`, `1` photon for dust RT)

Neither run uses the Charlot & Fall birth-cloud dust (`CF_on = False` in both masters); the two parameter
masters are identical except the dust-grid type, the dust-to-metals ratio, and the dust-raytracing photon count.
Outputs go to a fresh `sed_dust_comparison/` folder so earlier runs are untouched.
Target sample: post-starburst galaxies from SIMBA-100.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from astropy.cosmology import LambdaCDM
import astropy.units as u
from astropy.io import fits
from astropy.table import Table
from scipy import interpolate

from simbanator.sed.makesed import MakeSED
from simbanator.io.simba import Simulation

cosmo = LambdaCDM(H0=68, Om0=0.3, Ode0=0.7, Ob0=0.048)
plt.rcParams.update({'font.size': 14})

sim = Simulation('cis100')

### Configuration — shared sample + the two dust runs

Both runs read the identical `(snap, galaxyID)` source list and write to **separate** `run_tag`
output trees, so they never overwrite each other. Each run only differs in which parameter master
it copies into the Powderday job dirs.

In [ ]:
GVFS_BASE   = ''#"/run/user/1000/gvfs/sftp:host=slurmusrint2.cis.gov.pl,user=glorenzon"
# Use '+' not os.path.join: with GVFS_BASE='' this must stay ABSOLUTE ('/mnt/...'), and
# os.path.join(GVFS_BASE, "mnt/...") yields a RELATIVE path that doubles against the CWD
# (e.g. .../analize_simba_cgm/mnt/home/glorenzon/...), so Stage 2 can't find the SEDs.
REMOTE_HOME = GVFS_BASE + "/mnt/home/glorenzon/analize_simba_cgm"

hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
selection_file = 'selection_simba100_poststarburst.hdf5'
# fresh output folder so this dust-comparison experiment never overwrites previous runs (e.g. .../sed/PSBG_dust_off)
output_dir     = os.path.join(REMOTE_HOME, 'output', sim.name, 'sed_dust_comparison')
SAMPLE_FITS    = '/mnt/home/glorenzon/analize_simba_cgm/output/cis100/quenching_times/forward_modeled_unique_sample.fits'

# filtered-particle filename prefix — MUST be the same value passed to extract_particles (Stage 0)
# and create_master (Stage 1). Stage 0 saves <PREFIX>_snap<NNN>_gal<IDIDID>.h5; create_master bakes
# the identical name into each snap_*.py via the cluster template, so the two must never drift.
PARTICLE_PREFIX = 'm100n1024'

# the two runs: identical in every respect except the parameter master (dust treatment)
RUNS = {
    'dust_on':  dict(run_tag='dusty_simdust',  paramf='parameters_master.py'),
    'dust_off': dict(run_tag='nodust_1e-12',   paramf='parameters_master-nodust.py'),
}

# one MakeSED per run: same sim / hydro dir / selection file / output_dir, distinct run_tag
makeseds = {
    key: MakeSED(sim, nnodes=1, model_run_name=cfg['run_tag'],
                 hydro_dir_base=hydro_dir_base, selection_file=selection_file,
                 output_dir=output_dir, run_tag=cfg['run_tag'])
    for key, cfg in RUNS.items()
}
for key, ms in makeseds.items():
    print(f"{key:9s} -> run_tag='{ms.run_tag}', master='{RUNS[key]['paramf']}'")

### Shared source list (identical for both runs)

Read the PSB sample **once** and reuse the same `(SNAPS, IDS)` arrays for selection, master
generation, and flux extraction in both runs — this is what guarantees the two runs contain
exactly the same galaxies.

In [ ]:
with fits.open(SAMPLE_FITS) as f:
    _snaps = np.asarray(f[1].data['SNAPSHOT'], dtype=int)
    _ids   = np.asarray(f[1].data['GROUPID_SNAPSHOT'], dtype=int)

# sort + de-duplicate (snap, id) so the source list is canonical and identical for both runs
_pairs = np.unique(np.stack([_snaps, _ids], axis=1), axis=0)
SNAPS, IDS = _pairs[:, 0], _pairs[:, 1]

_ndup = len(_snaps) - len(SNAPS)
print(f"{len(SNAPS)} unique sources over {len(np.unique(SNAPS))} snapshots"
      + (f"  ({_ndup} duplicate pair(s) removed)" if _ndup else ""))

### Stage 0 — extract the shared particle files (once, used by both runs)

Powderday reads per-galaxy particle files from `hydro_dir_base/snap_NNN/`. They are **identical for
`dust_on` and `dust_off`** — the dust treatment is set by the parameter master, not the particle
data — so extract them **once** here, before Stage 1. `extract_particles` reads each snapshot a
single time, writes one HDF5 per galaxy (gas + stars; gas keeps its `Dust_Masses` field), and skips
files that already exist unless `EXTRACT_OVERWRITE=True`.

In [ ]:
from simbanator.analysis import extract_particles

# Stage 0 — extract the per-galaxy particle files ONCE. Powderday reads these from
# hydro_dir_base/snap_NNN/<PARTICLE_PREFIX>_snap<snap>_gal<id>.h5; they are IDENTICAL for dust_on and
# dust_off (the dust treatment lives in the parameter master, not the particles), so both runs share
# them. extract_particles writes under cwd/output/<sim>/filtered_particles == hydro_dir_base, one
# file per galaxy, reading each snapshot only once. Existing files are skipped unless overwrite=True.
#
# The prefix MUST match create_master's prefix (Stage 1): it is the value baked into each snap_*.py
# as snapshot_name. Both use PARTICLE_PREFIX so the saved name and the looked-up name can't diverge.
#
# Truncated/corrupt snapshot files (a half-staged download) raise OSError; we skip that snapshot,
# keep going, and list the casualties at the end so the run isn't lost to one bad file.
EXTRACT_OVERWRITE = False
EXTRACT_PTYPES    = ("PartType0", "PartType4")   # gas (carries Dust_Masses) + stars; add "PartType5" for AGN

bad_snaps = []                                    # (snap, n_gal, error) for unreadable snapshots
for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    try:
        _cs = sim.load_catalog(snap=_snap)
        extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, ptypes=EXTRACT_PTYPES,
                          sim_name=sim.name, prefix=PARTICLE_PREFIX,
                          overwrite=EXTRACT_OVERWRITE, verbose=1)
        del _cs
    except (OSError, KeyError) as e:
        print(f"  [SKIP] snap {_snap}: {type(e).__name__}: {str(e).splitlines()[0]}")
        bad_snaps.append((_snap, len(_ids_here), f"{type(e).__name__}: {str(e).splitlines()[0]}"))

print("\nparticle extraction complete ->", hydro_dir_base)
if bad_snaps:
    n_gal_bad = sum(n for _, n, _ in bad_snaps)
    print(f"\n{len(bad_snaps)} snapshot(s) unreadable -> {n_gal_bad} galaxies skipped "
          f"(these (snap, id) pairs will be missing from BOTH runs, consistently):")
    for s, n, err in bad_snaps:
        ids_s = np.unique(IDS[SNAPS == s])
        print(f"  snap {s:3d}: {n} galaxies {list(ids_s)} | {err}")
    print("\nThese snapshot files are incomplete on disk (truncated). Re-stage/re-download them in")
    print("the shared SIMBA data dir, then re-run this cell (existing files are skipped).")

### Stage 1 — write the HDF5 selection and Slurm masters (both runs)

Run this, then launch the Powderday RT jobs on the cluster (`submit_all_snaps.sh` under each run's
`powderday_sed_out/`). Come back and run Stage 2 once the `.rtout.sed` files exist.

In [ ]:
for key, cfg in RUNS.items():
    ms = makeseds[key]
    print(f"\n=== {key} (run_tag='{ms.run_tag}', master='{cfg['paramf']}') ===")
    ms.selection_gals(snaps=SNAPS, galaxyID=IDS)                 # same sources for every run
    ms.create_master('cluster', 'plist', radius=None, partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
                     prefix=PARTICLE_PREFIX, paramf=cfg['paramf'], snaps_to_run=None)  # same prefix Stage 0 saved with

### Filter set for flux extraction

`local_filters` supplements the built-in filter library; each entry is keyed as
`{telescope_label: {instrument_label: {filter_name: path_to_res_file}}}`.
Johnson V and U must use distinct top-level keys (`Johnson`, `Johnson2`) because a single
instrument cannot hold two identically-named sub-entries in this dict.

In [ ]:
local_filters = {
    '2MASS': {
        'J': {
            'J': '/mnt/home/glorenzon/analize_simba_cgm/2MASS_J.res',
        }
    },
    'Johnson': {
        'V': {
            'V': '/mnt/home/glorenzon/analize_simba_cgm/maiz-apellaniz_Johnson_V.res',
        }
    },
    # separate top-level key required: dict cannot hold two entries under 'Johnson'
    'Johnson2': {
        'U': {
            'U': '/mnt/home/glorenzon/analize_simba_cgm/maiz-apellaniz_Johnson_U.res',
        }
    },
}

### Stage 2 — extract photometric fluxes (run after RT completes)

Extracts fluxes for the **same** `(SNAPS, IDS)` in both runs. `extract_flux_batch` prints a
`requested / written / dropped` summary and writes `missing_sources.txt` listing any galaxy whose
`.rtout.sed` is absent or unreadable — so a count mismatch between the two runs is never silent.

In [ ]:
flux_files = {}
for key in RUNS:
    ms = makeseds[key]
    print(f"\n=== extracting fluxes: {key} (run_tag='{ms.run_tag}') ===")
    flux_file, xmean_file = ms.extract_flux_batch(
        SNAPS, IDS,
        ['HST', 'JWST', 'Spitzer', 'Herschel'],
        ['WFC3', 'NIRCam', 'IRAC', 'SPIRE'],
        filters=None, local_filters=local_filters, wave_unit='micron', findx=0,
    )
    flux_files[key] = flux_file

### Cross-check — both runs contain exactly the same sources

If the two runs disagree, the offending `(snap, gal)` pairs are listed here and in each run's
`missing_sources.txt`; chase those `.rtout.sed` files on the cluster (a 1-photon dust run can
crash or truncate output for some galaxies).

In [ ]:
present = {}
for key, ff in flux_files.items():
    t = Table.read(ff)
    present[key] = set(zip(np.asarray(t['snap'], int), np.asarray(t['gal_id_at_snap'], int)))

on, off = present['dust_on'], present['dust_off']
print(f"dust_on={len(on)}  dust_off={len(off)}  common={len(on & off)}")
print("only in dust_on :", sorted(on - off))
print("only in dust_off:", sorted(off - on))
print("OK — both runs contain exactly the same sources." if on == off
      else "MISMATCH — see the lists above and each run's missing_sources.txt")

### Photometric SEDs — dust_on vs dust_off

Per-galaxy multi-band fluxes vs effective filter wavelength, one panel per run (shared axes).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from astropy.table import Table

meta_cols = {'gal_id_at_snap', 'snap', 'redshift'}

# all comparison figures land in their own subfolder, alongside (not inside) the two run trees
PLOT_DIR = os.path.join(os.path.commonpath(list(flux_files.values())), 'sed_plots')
os.makedirs(PLOT_DIR, exist_ok=True)


def load_run_seds(key):
    """One run's photometry as (wave[um], flux[N_gal, N_filt] mJy, filter_names, z, (snap, gal) keys),
    filters sorted by effective wavelength (median xmean)."""
    ftab = Table.read(flux_files[key])
    xtab = Table.read(os.path.join(os.path.dirname(flux_files[key]), 'all_xmean.fits'))
    fcols = [c for c in ftab.colnames if c not in meta_cols]
    xmean = {f: np.nanmedian(xtab['xmean'][xtab['filter'] == f]) for f in fcols}
    order = sorted(fcols, key=lambda f: xmean[f])
    wave  = np.array([xmean[f] for f in order], float)
    flux  = np.array([[float(row[f]) for f in order] for row in ftab], float)
    z     = np.asarray(ftab['redshift'], float)
    keys  = list(zip([int(s) for s in ftab['snap']], [int(g) for g in ftab['gal_id_at_snap']]))
    return wave, flux, order, z, keys


def label_bands(ax, wave, names, min_dex=0.045):
    """Faint guide line at every filter; band name only where it won't overlap the previous label."""
    last = -np.inf
    for w, nm in zip(wave, names):
        ax.axvline(w, color='0.85', lw=0.6, zorder=0)
        if np.log10(w) - last >= min_dex:
            ax.annotate(nm.split('.')[-1], xy=(w, 1.005), xycoords=('data', 'axes fraction'),
                        rotation=90, va='bottom', ha='center', fontsize=7, color='0.45',
                        annotation_clip=False)
            last = np.log10(w)


plt.rcParams.update({'font.size': 13, 'axes.linewidth': 1.1})
runs = ['dust_on', 'dust_off']

# shared redshift colour scale across both panels
all_z = np.concatenate([load_run_seds(k)[3] for k in runs])
norm  = Normalize(vmin=np.nanmin(all_z), vmax=np.nanmax(all_z))
cmap  = plt.cm.viridis

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, key in zip(axes, runs):
    wave, flux, names, z, _ = load_run_seds(key)
    for i in range(flux.shape[0]):                       # per-galaxy SED, coloured by redshift
        ax.plot(wave, flux[i], color=cmap(norm(z[i])), lw=0.6, alpha=0.45, zorder=1)
    med = np.nanmedian(flux, axis=0)
    lo, hi = np.nanpercentile(flux, [16, 84], axis=0)
    ax.fill_between(wave, lo, hi, color='0.6', alpha=0.30, zorder=2, label='16-84%')
    ax.plot(wave, med, color='k', lw=2.5, zorder=3, label='median')

    ax.set(xscale='log', yscale='log')
    ax.set_xlabel(r'$\lambda_\mathrm{eff}\ [\mu\mathrm{m}]$')
    ax.grid(which='both', ls=':', alpha=0.4)
    ax.legend(frameon=False, loc='lower center')
    ax.annotate(f"{key}   (N = {flux.shape[0]})", xy=(0.03, 0.96), xycoords='axes fraction',
                fontweight='bold', va='top', ha='left',
                bbox=dict(boxstyle='round', fc='w', ec='0.7', alpha=0.85))
    label_bands(ax, wave, names)                         # light, non-crowding band labels

axes[0].set_ylabel('Flux [mJy]')
sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cb = fig.colorbar(sm, ax=axes, pad=0.01, fraction=0.045)
cb.set_label('redshift $z$')
fig.suptitle('Photometric SEDs per galaxy — dust on vs dust off', fontweight='bold')

out = os.path.join(PLOT_DIR, 'sed_per_galaxy_dust_on_off')
fig.savefig(out + '.png', dpi=200, bbox_inches='tight')
fig.savefig(out + '.pdf', bbox_inches='tight')
print('saved ->', out + '.png')
plt.show()

### Dust on vs dust off — matched galaxies

Directly compares the **same** galaxies under the two dust treatments. The top panel overlays the
median SED (with 16–84% scatter) of each run; the bottom panel is the per-galaxy attenuation curve
$A_\lambda = -2.5\log_{10}(F_\mathrm{on}/F_\mathrm{off})$ — positive where dust absorbs starlight
(UV/optical), crossing to negative where reprocessed dust *emission* dominates (mid/far-IR).
All figures are written to the shared `sed_plots/` subfolder (PNG + PDF).

In [ ]:
# Dust ON vs OFF for the SAME galaxies — median SEDs (top) and the per-galaxy attenuation
# curve A_lambda = -2.5 log10(F_on / F_off): positive where dust absorbs (UV/optical),
# negative where reprocessed dust emission dominates (mid/far-IR).
wOn,  fOn,  nOn,  zOn,  kOn  = load_run_seds('dust_on')
wOff, fOff, nOff, zOff, kOff = load_run_seds('dust_off')

# shared filters (sorted by wavelength) and galaxies present in BOTH runs
common_filt = [n for n in nOn if n in set(nOff)]
wave = np.array([wOn[nOn.index(n)] for n in common_filt])
iOn  = [nOn.index(n)  for n in common_filt]
iOff = [nOff.index(n) for n in common_filt]

on_by  = {k: fOn[r][iOn]   for r, k in enumerate(kOn)}
off_by = {k: fOff[r][iOff] for r, k in enumerate(kOff)}
common_gal = [k for k in kOn if k in off_by]                 # preserve dust_on order
on_mat  = np.array([on_by[k]  for k in common_gal])
off_mat = np.array([off_by[k] for k in common_gal])
print(f"{len(common_gal)} galaxies common to both runs, {len(common_filt)} shared filters")

# per-galaxy attenuation in magnitudes, then median + 16-84% band across galaxies
with np.errstate(divide='ignore', invalid='ignore'):
    A = -2.5 * np.log10(on_mat / off_mat)
A[~np.isfinite(A)] = np.nan

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]},
                               constrained_layout=True)

for mat, c, lab in [(on_mat, 'C3', 'dust on'), (off_mat, 'C0', 'dust off')]:
    med = np.nanmedian(mat, axis=0)
    lo, hi = np.nanpercentile(mat, [16, 84], axis=0)
    ax1.fill_between(wave, lo, hi, color=c, alpha=0.20)
    ax1.plot(wave, med, color=c, lw=2.5, marker='o', ms=4, label=lab)
ax1.set(xscale='log', yscale='log', ylabel='Flux [mJy]')
ax1.grid(which='both', ls=':', alpha=0.4)
ax1.legend(frameon=False, title=f'same {len(common_gal)} galaxies')
ax1.set_title('Dust on vs off — matched galaxies', fontweight='bold')

medA = np.nanmedian(A, axis=0)
loA, hiA = np.nanpercentile(A, [16, 84], axis=0)
ax2.fill_between(wave, loA, hiA, color='0.5', alpha=0.30)
ax2.plot(wave, medA, color='k', lw=2.5, marker='o', ms=4)
ax2.axhline(0, color='0.7', lw=1, ls='--')
ax2.set(xscale='log', xlabel=r'$\lambda_\mathrm{eff}\ [\mu\mathrm{m}]$',
        ylabel=r'$-2.5\,\log_{10}(F_\mathrm{on}/F_\mathrm{off})$ [mag]')
ax2.grid(which='both', ls=':', alpha=0.4)
ax2.annotate('attenuated\n(UV/opt)', xy=(0.02, 0.93), xycoords='axes fraction', fontsize=9, va='top')
ax2.annotate('dust emission (IR)', xy=(0.98, 0.10), xycoords='axes fraction', fontsize=9, ha='right')

out = os.path.join(PLOT_DIR, 'sed_dust_on_vs_off_attenuation')
fig.savefig(out + '.png', dpi=200, bbox_inches='tight')
fig.savefig(out + '.pdf', bbox_inches='tight')
print('saved ->', out + '.png')
plt.show()